# Laboratorio 4 - Segmentación de imágenes con técnicas de agrupación
## Caso: DermAlpes

**Curso:** ISIS2611 - Machine Learning

**Integrantes:**
- Gabriela Hernández Roncancio - 202412025
- Julián Donato Gutiérrez- 202415588

## Tabla de contenido

1. [Importación de librerías](#1)
2. [Comprensión del problema](#2)
3. [Carga de los datos](#3)
4. [Exploración de las imágenes de ajuste](#4)
5. [Pipeline de preparación y representación (Actividad 1)](#5)
6. [Agrupación y selección de K (Actividad 2)](#6)
7. [Construcción de la máscara de la lesión (Actividad 3)](#7)
8. [Ajuste del método (Actividad 4)](#8)
9. [Evaluación sobre las imágenes de prueba (Actividad 5)](#9)
10. [Video (Actividad 6)](#10)
11. [Análisis de resultados](#11)
12. [Uso de herramientas de IA generativa](#12)
13. [Nota sobre la reproducibilidad de los resultados](#13)
14. [Referencias](#14)

> **Nota:** este notebook es una guía. Los textos en cursiva dicen qué hay que hacer o decidir en cada sección y deben reemplazarse por sus propias explicaciones. El código está comentado como **sugerencia** (basado en las prácticas de K-medias y de segmentación de imágenes): es una forma posible de hacerlo, no la única. Para usar una celda, selecciónenla completa y descoméntenla con `Cmd + /` (o `Ctrl + /`).

## <a name="1"></a> 1. Importación de librerías

_Importen las librerías que vayan a usar. El enunciado menciona NumPy, Scikit-learn, Pillow / scikit-image / OpenCV para leer las imágenes y Matplotlib / Seaborn._

**Sugerencia:** la celda de abajo usa OpenCV, como la práctica de segmentación, e imprime las versiones como la práctica de K-medias. Ajústenla a lo que terminen usando.

> Si no tienen OpenCV instalado: `pip install opencv-python`.

In [ ]:
# import os
# import time
# import cv2
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from glob import glob
# from importlib.metadata import version
#
# from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.pipeline import Pipeline
# from sklearn.cluster import KMeans
# from sklearn.metrics import (silhouette_score, f1_score, precision_score,
#                              recall_score, v_measure_score)
#
# print(f"Versión de OpenCV: {cv2.__version__}")
# print(f"Versión de Numpy: {version('numpy')}")
# print(f"Versión de Pandas: {version('pandas')}")
# print(f"Versión de Matplotlib: {version('matplotlib')}")
# print(f"Versión de Scikit-learn: {version('scikit-learn')}")

**Sugerencia:** definir en una sola celda las constantes que controlan la reproducibilidad y el costo computacional (semilla, tamaño de la muestra para la silueta, tamaño de las imágenes, rutas). Los valores de abajo son solo un punto de partida: decidan los suyos y justifíquenlos más adelante. El enunciado pide documentar el tamaño de la muestra y la semilla.

In [ ]:
# # Semilla para K-medias y para la muestra de píxeles de la silueta
# SEMILLA = 0
#
# # Número de píxeles de la muestra con que se calcula el coeficiente de silueta
# TAMANO_MUESTRA_SILUETA = 3000
#
# # Ancho (en píxeles) al que se redimensionan las imágenes; el alto se calcula conservando la proporción
# ANCHO = 256
#
# # Valores de K que se exploran
# VALORES_K = list(range(2, 9))
#
# # Rutas de los datos (ajustarlas según cómo se descomprima el conjunto de datos)
# DIR_IMAGENES = os.path.join('.', 'data', 'imagenes')
# DIR_MASCARAS = os.path.join('.', 'data', 'mascaras')
#
# # Partición definida en el enunciado
# IDS_AJUSTE = ['ISIC_0000085', 'ISIC_0000086', 'ISIC_0000089', 'ISIC_0000094', 'ISIC_0000108',
#               'ISIC_0000114', 'ISIC_0000116', 'ISIC_0000122', 'ISIC_0000133', 'ISIC_0000137',
#               'ISIC_0000150', 'ISIC_0000153', 'ISIC_0000156', 'ISIC_0000157', 'ISIC_0000112']
# IDS_PRUEBA = ['ISIC_0000087', 'ISIC_0000105', 'ISIC_0000110', 'ISIC_0000123', 'ISIC_0000128']

## <a name="2"></a> 2. Comprensión del problema

_Expliquen con sus palabras el caso y qué se busca resolver._

Puntos del enunciado que conviene que queden claros:
- DermAlpes quiere una herramienta que **delimite la lesión** en fotos de dermatoscopio para que el especialista mida su tamaño y forma. **No hace diagnósticos.**
- No hay imágenes etiquetadas propias, por eso se prueba un enfoque **no supervisado**: agrupar los píxeles con K-medias según color y posición, y luego decidir qué grupos son la lesión.
- Las máscaras de referencia **no** se usan para agrupar, solo para evaluar.
- Protocolo: 15 imágenes de **ajuste** (para elegir β, λ, el criterio de la máscara y K) y 5 de **prueba** (se consultan solo al final).

_Preguntas guía: ¿qué sería un buen resultado para DermAlpes? ¿qué tipo de error creen que sería más costoso y por qué?_

## <a name="3"></a> 3. Carga de los datos

Fuente: ISIC 2016, tarea de segmentación de lesiones (Gutman et al., 2016). [Enlace](https://challenge.isic-archive.com/landing/2016/37/).

Cada imagen `ISIC_XXXXXXX.jpg` tiene una máscara `ISIC_XXXXXXX_Segmentation.png` con valores 0 (piel o fondo) y 255 (lesión).

**Qué pide el enunciado:**
- Convertir las máscaras de {0, 255} a {0, 1}.
- La máscara estimada y la de referencia deben tener las mismas dimensiones. Si reducen las imágenes, las máscaras se redimensionan con **vecino más cercano** para que sigan siendo binarias.
- Citar la fuente del conjunto de datos.

**Decisiones que tienen que tomar y justificar:**
- _¿Van a reducir el tamaño de las imágenes? ¿A cuánto?_ Piensen en el número de píxeles (instancias) por imagen y en cuántas veces se va a correr K-medias: 15 imágenes × 10 combinaciones × 7 valores de K. Reducir mucho ahorra tiempo pero se pierde detalle del borde de la lesión.
- _¿Qué interpolación usar para la imagen?_ (por ejemplo, `INTER_AREA` promedia píxeles al reducir).
- Ojo: OpenCV lee en BGR, así que hay que convertir a RGB.

**Sugerencia:** el código de abajo es una opción. Reduce cada imagen a `ANCHO` píxeles de ancho conservando la proporción, y la máscara al mismo tamaño con `INTER_NEAREST`.

In [ ]:
# def redimensionar(img, ancho, interpolacion):
#     """Redimensiona una imagen al ancho indicado conservando la proporción alto/ancho."""
#     alto = round(img.shape[0] * ancho / img.shape[1])
#     return cv2.resize(img, (ancho, alto), interpolation=interpolacion)
#
#
# def cargar_imagen(id_img):
#     """Carga una imagen en RGB y la redimensiona. Retorna la imagen y su tamaño original."""
#     ruta = glob(os.path.join(DIR_IMAGENES, f'{id_img}.*'))[0]
#     img = cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2RGB)
#     return redimensionar(img, ANCHO, cv2.INTER_AREA), img.shape[:2]
#
#
# def cargar_mascara(id_img):
#     """Carga la máscara de referencia, la redimensiona por vecino más cercano y la convierte a {0, 1}."""
#     ruta = glob(os.path.join(DIR_MASCARAS, f'{id_img}_Segmentation.*'))[0]
#     mascara = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
#     mascara = redimensionar(mascara, ANCHO, cv2.INTER_NEAREST)
#     return (mascara == 255).astype(np.uint8)

**Sugerencia de protocolo:** cargar aquí solo las máscaras de **ajuste** y dejar las de prueba para la sección 9, cuando la configuración ya esté fija. Así no se pueden usar por accidente durante el ajuste.

In [ ]:
# imagenes, tamanos_originales = {}, {}
# for id_img in IDS_AJUSTE + IDS_PRUEBA:
#     imagenes[id_img], tamanos_originales[id_img] = cargar_imagen(id_img)
#
# mascaras_ajuste = {id_img: cargar_mascara(id_img) for id_img in IDS_AJUSTE}
#
# print(f'Imágenes cargadas: {len(imagenes)}')
# print(f'Máscaras de ajuste cargadas: {len(mascaras_ajuste)}')

**Sugerencia:** comprobar que las máscaras quedaron binarias y del mismo tamaño que su imagen.

In [ ]:
# for id_img, mascara in mascaras_ajuste.items():
#     assert mascara.shape == imagenes[id_img].shape[:2], f'Dimensiones distintas en {id_img}'
#     assert set(np.unique(mascara)) <= {0, 1}, f'Máscara no binaria en {id_img}'
# print('Todas las máscaras de ajuste son binarias y tienen el tamaño de su imagen.')

## <a name="4"></a> 4. Exploración de las imágenes de ajuste

_Esta sección no la pide el enunciado explícitamente, pero les va a servir para justificar el criterio de la máscara (Actividad 3) y para explicar los casos de fallo._

**Ideas de qué revisar:**
- Tamaño original de cada imagen y qué proporción ocupa la lesión.
- Cada imagen con el contorno de su máscara de referencia.
- Casos que pueden ser difíciles: lesiones claras o poco contrastadas, esquinas oscuras del dermatoscopio, vellos, burbujas, reglas o marcas.

In [ ]:
# resumen_datos = pd.DataFrame([{
#     'id': id_img,
#     'tamaño original (alto x ancho)': f'{tamanos_originales[id_img][0]} x {tamanos_originales[id_img][1]}',
#     'tamaño usado (alto x ancho)': f'{imagenes[id_img].shape[0]} x {imagenes[id_img].shape[1]}',
#     'píxeles': imagenes[id_img].shape[0] * imagenes[id_img].shape[1],
#     '% lesión': round(100 * mascaras_ajuste[id_img].mean(), 1),
# } for id_img in IDS_AJUSTE])
# resumen_datos

In [ ]:
# def mostrar_con_contorno(ids, mascaras, titulo, cols=5):
#     """Muestra las imágenes con el contorno de su máscara de referencia en amarillo."""
#     rows = int(np.ceil(len(ids) / cols))
#     fig, ax = plt.subplots(rows, cols, figsize=(3.2 * cols, 2.6 * rows), squeeze=False)
#     for a in ax.flat:
#         a.axis('off')
#     for a, id_img in zip(ax.flat, ids):
#         a.imshow(imagenes[id_img])
#         if mascaras is not None:
#             a.contour(mascaras[id_img], levels=[0.5], colors='yellow', linewidths=1.2)
#         a.set_title(id_img, fontsize=9)
#     plt.suptitle(titulo)
#     plt.tight_layout()
#     plt.show()
#
#
# mostrar_con_contorno(IDS_AJUSTE, mascaras_ajuste, 'Imágenes de ajuste con el contorno de la máscara de referencia')

**Opcional:** en la práctica de segmentación se graficaron los píxeles en el espacio RGB (`plot_data`, con Plotly en 3D). Algo parecido sirve aquí para ver si la lesión y la piel se separan solo por color. La celda de abajo es una versión en 2D que colorea una muestra de píxeles según la máscara de referencia (solo para imágenes de ajuste).

In [ ]:
# def graficar_espacio_color(id_img, n=3000):
#     """Grafica una muestra de píxeles en los planos R-G y R-B, coloreados según la máscara de referencia."""
#     rng = np.random.default_rng(SEMILLA)
#     X = imagenes[id_img].reshape(-1, 3) / 255.0
#     y = mascaras_ajuste[id_img].ravel()
#     idx = rng.choice(len(X), size=min(n, len(X)), replace=False)
#
#     fig, ax = plt.subplots(1, 3, figsize=(14, 4))
#     ax[0].imshow(imagenes[id_img])
#     ax[0].axis('off')
#     ax[0].set_title(id_img)
#     for a, (c1, c2, n1, n2) in zip(ax[1:], [(0, 1, 'R', 'G'), (0, 2, 'R', 'B')]):
#         for valor, color, nombre in [(0, 'tab:blue', 'piel/fondo'), (1, 'tab:red', 'lesión')]:
#             sel = idx[y[idx] == valor]
#             a.scatter(X[sel, c1], X[sel, c2], s=3, alpha=0.4, color=color, label=nombre)
#         a.set(xlabel=n1, ylabel=n2)
#         a.legend()
#         a.grid()
#     plt.tight_layout()
#     plt.show()
#
#
# # Cambiar el identificador para explorar otras imágenes
# graficar_espacio_color(IDS_AJUSTE[0])

**Conclusiones de la exploración:**

_Escriban aquí sus conclusiones. Preguntas guía: ¿qué imágenes tienen lesiones claras o poco contrastadas? ¿cuáles tienen esquinas oscuras, vellos, reglas o burbujas? ¿qué porcentaje de la imagen ocupa la lesión en promedio? ¿qué dificultades anticipan para K-medias?_

## <a name="5"></a> 5. Pipeline de preparación y representación (Actividad 1)

**Qué pide el enunciado:** un **pipeline de Scikit-learn** que convierta cada imagen en una matriz con un píxel por fila y las características

$$[\beta \cdot R,\ \beta \cdot G,\ \beta \cdot B,\ \lambda \cdot x,\ \lambda \cdot y]$$

con color y coordenadas normalizados a $[0, 1]$. β y λ deben poder cambiarse **sin modificar la implementación**.

**Decisiones que tienen que tomar y justificar:**
- _¿Cómo normalizar el color?_ Opciones: dividir por 255 (escala fija) o un `MinMaxScaler` por imagen, como en la práctica de K-medias. Piensen qué le pasa al peso relativo de los canales en cada caso.
- _¿Cómo normalizar las coordenadas?_ Por ejemplo, $x = j / (W - 1)$ y $y = i / (H - 1)$.
- _¿Cómo hacer que β y λ sean parámetros del pipeline?_ Una opción es escribir transformadores propios (`BaseEstimator`, `TransformerMixin`) y cambiarlos con `set_params`, como `model__n_clusters` en la práctica de K-medias.
- Relación con la práctica de segmentación: allí la posición se agregaba como coordenadas sin normalizar multiplicadas por `weight`. ¿Qué cambia si las coordenadas están en $[0, 1]$?

**Sugerencia:** el código de abajo implementa una opción. Usa dos transformadores, `ImagenAPixeles` (divide el color por 255 y agrega $x, y$ normalizados, con el `meshgrid` de la práctica de segmentación) y `PonderarColorPosicion` (multiplica por β y λ), seguidos de `KMeans`.

_Escriban aquí su justificación del pipeline._

In [ ]:
# class ImagenAPixeles(BaseEstimator, TransformerMixin):
#     """Convierte una imagen RGB (alto, ancho, 3) en una matriz con un píxel por fila: [R, G, B, x, y] en [0, 1]."""
#
#     def fit(self, X, y=None):
#         return self
#
#     def transform(self, X):
#         alto, ancho = X.shape[:2]
#         I, J = np.meshgrid(np.arange(alto), np.arange(ancho), indexing='ij')
#         color = X.reshape(-1, 3) / 255.0
#         x = J.reshape(-1, 1) / (ancho - 1)
#         y = I.reshape(-1, 1) / (alto - 1)
#         return np.hstack([color, x, y])
#
#
# class PonderarColorPosicion(BaseEstimator, TransformerMixin):
#     """Multiplica las columnas de color por beta y las de posición por lam."""
#
#     def __init__(self, beta=1.0, lam=0.0):
#         self.beta = beta
#         self.lam = lam
#
#     def fit(self, X, y=None):
#         if self.beta == 0 and self.lam == 0:
#             raise ValueError('beta y lam no pueden ser ambos 0')
#         return self
#
#     def transform(self, X):
#         pesos = np.array([self.beta] * 3 + [self.lam] * 2)
#         return X * pesos

In [ ]:
# kmeans = KMeans(n_init=3, random_state=SEMILLA)
#
# pipe = Pipeline(steps=[("pixeles", ImagenAPixeles()),
#                        ("pesos", PonderarColorPosicion()),
#                        ("model", kmeans)])
# pipe

**Sugerencia:** probar la representación en una imagen. Con β = λ = 1 todas las columnas deberían quedar en $[0, 1]$; con otros valores, cada columna debería quedar multiplicada por su peso.

In [ ]:
# COLUMNAS = ['β·R', 'β·G', 'β·B', 'λ·x', 'λ·y']
# img_ejemplo = imagenes[IDS_AJUSTE[0]]
#
# for beta, lam in [(1, 1), (2, 0.5)]:
#     pipe.set_params(pesos__beta=beta, pesos__lam=lam)
#     X_ejemplo = pd.DataFrame(pipe[:-1].transform(img_ejemplo), columns=COLUMNAS)
#     print(f'beta = {beta}, lam = {lam} -> matriz de {X_ejemplo.shape[0]} píxeles x {X_ejemplo.shape[1]} características')
#     display(X_ejemplo.describe().loc[['min', 'max']].round(3))

## <a name="6"></a> 6. Agrupación y selección de K (Actividad 2)

### 6.1 Combinaciones de β y λ

**Qué pide el enunciado:** diez combinaciones de β y λ con **razones λ/β distintas**, incluyendo una de solo color (λ = 0, β ≠ 0) y una de solo posición (β = 0, λ ≠ 0). Lo que importa es la razón: (β = 1, λ = 0.5) y (β = 2, λ = 1) dan los mismos grupos.

**Decisión que tienen que tomar:** _¿qué razones van a explorar y por qué ese rango?_ Pistas: con razones pequeñas domina el color, y con razones grandes los grupos tienden a ser bloques espaciales que ignoran el color. Pueden apoyarse en lo que vieron en la exploración.

**Sugerencia:** la tabla de abajo es solo un ejemplo de rango (β = 1 fijo y λ variando). Cámbienla por sus propias combinaciones.

_Escriban aquí la justificación de sus combinaciones._

In [ ]:
# COMBINACIONES = pd.DataFrame({
#     'beta': [1, 0, 1,   1,    1,   1,    1, 1,   1, 1],
#     'lam':  [0, 1, 0.1, 0.25, 0.5, 0.75, 1, 1.5, 2, 4],
# }, index=[f'C{i}' for i in range(10)])
# COMBINACIONES.index.name = 'combinacion'
# COMBINACIONES['lam/beta'] = np.where(COMBINACIONES['beta'] == 0, np.inf, COMBINACIONES['lam'] / COMBINACIONES['beta'])
# COMBINACIONES['descripcion'] = np.select(
#     [COMBINACIONES['lam'] == 0, COMBINACIONES['beta'] == 0], ['solo color', 'solo posición'], 'color + posición')
# COMBINACIONES

### 6.2 Selección de K con el coeficiente de silueta

**Qué pide el enunciado:** para cada imagen de ajuste y cada combinación, aplicar K-medias con K entre 2 y 8 y elegir K con el **coeficiente de silueta**, calculado sobre una **muestra reproducible** de píxeles. Hay que documentar el tamaño de la muestra y la semilla.

**Decisiones que tienen que tomar y justificar:**
- _¿De qué tamaño es la muestra?_ La silueta calcula distancias entre todos los pares de puntos ($O(n^2)$), así que con todos los píxeles sería muy lenta. Una muestra más grande es más estable pero más costosa.
- _¿Cómo hacerla reproducible?_ `silhouette_score` tiene los parámetros `sample_size` y `random_state`. Piensen si conviene usar la misma muestra para todos los K de una imagen, para que las siluetas sean comparables.
- _¿Sobre qué datos calcular la silueta?_ En la práctica de K-medias se calculaba sobre los datos ya transformados (`pipe[:-1].transform(...)`).

**Documenten aquí sus valores:**
- Tamaño de la muestra: ___
- Semilla: ___

**Sugerencia:** la función de abajo sigue la idea de `evaluar_k` de la práctica de K-medias. Además guarda las etiquetas de todos los K, que sirven después para la pregunta 2 del análisis.

In [ ]:
# def seleccionar_k(img, beta, lam, valores_k=VALORES_K):
#     """
#     Aplica K-medias para cada K y selecciona el de mayor silueta.
#
#     Parametros:
#     img : np.ndarray
#         Imagen RGB redimensionada
#     beta, lam : float
#         Pesos del color y de la posición
#     valores_k : list
#         Valores de K a evaluar
#
#     Retorna un diccionario con el K seleccionado, su silueta, las siluetas e inercias de cada K
#     y las etiquetas de cada K (para poder analizar después la relación entre K, silueta y Dice).
#     """
#     pipe.set_params(pesos__beta=beta, pesos__lam=lam)
#     X = pipe[:-1].transform(img)
#     siluetas, inercias, etiquetas = [], [], []
#     for k in valores_k:
#         # Modificamos el parámetro k y ejecutamos el pipeline completo
#         pipe.set_params(model__n_clusters=k)
#         pipe.fit(img)
#         labels = pipe['model'].labels_
#         inercias.append(pipe['model'].inertia_)
#         siluetas.append(silhouette_score(X, labels, sample_size=TAMANO_MUESTRA_SILUETA, random_state=SEMILLA))
#         etiquetas.append(labels.astype(np.int8))
#     mejor = int(np.argmax(siluetas))
#     return {'k': valores_k[mejor], 'silueta': siluetas[mejor], 'siluetas': siluetas,
#             'inercias': inercias, 'etiquetas': etiquetas, 'labels': etiquetas[mejor]}

**Sugerencia:** antes de correr todo, mirar la inercia y la silueta en una sola imagen, como en la práctica de K-medias. Por ejemplo, comparando una combinación de solo color con una de color + posición (cambien la imagen y las combinaciones a su gusto).

In [ ]:
# def graficar_k(id_img, combinaciones):
#     """Grafica la inercia (método del codo) y el coeficiente de silueta para cada K y cada combinación."""
#     fig, axes = plt.subplots(1, 2, figsize=(13, 4))
#     for c in combinaciones:
#         beta, lam = COMBINACIONES.loc[c, ['beta', 'lam']]
#         res = seleccionar_k(imagenes[id_img], beta, lam)
#         etiqueta = f'{c} (β={beta}, λ={lam})'
#         axes[0].plot(VALORES_K, res['inercias'], marker='o', label=etiqueta)
#         axes[1].plot(VALORES_K, res['siluetas'], marker='o', label=etiqueta)
#         axes[1].scatter(res['k'], res['silueta'], s=150, facecolors='none', edgecolors='black')
#     axes[0].set(xlabel='Número de clusters', ylabel='Inercia', title='Método del codo')
#     axes[1].set(xlabel='Número de clusters', ylabel='Silhouette Score', title='Coeficiente de la silueta (círculo = K elegido)')
#     for ax in axes:
#         ax.set_xticks(VALORES_K)
#         ax.legend()
#         ax.grid()
#     plt.suptitle(id_img)
#     plt.tight_layout()
#     plt.show()
#
#
# graficar_k(IDS_AJUSTE[0], ['C0', 'C6'])

### 6.3 Ejecución sobre las 15 imágenes de ajuste

_Corran la selección de K para las 15 imágenes y todas sus combinaciones, y presenten el K elegido y su silueta._

**Sugerencia:** guardar los resultados en un diccionario por (imagen, combinación) para reutilizarlos en las siguientes secciones y no volver a correr K-medias. Esta celda puede tardar varios minutos.

In [ ]:
# resultados_k = {}
# inicio = time.time()
# for id_img in IDS_AJUSTE:
#     for c, fila in COMBINACIONES.iterrows():
#         resultados_k[(id_img, c)] = seleccionar_k(imagenes[id_img], fila['beta'], fila['lam'])
#     print(f'{id_img} listo ({time.time() - inicio:.0f} s acumulados)')

In [ ]:
# tabla_k = pd.DataFrame([{'id': id_img, 'combinacion': c, 'k': r['k'], 'silueta': r['silueta']}
#                         for (id_img, c), r in resultados_k.items()])
#
# print('K seleccionado por imagen y combinación:')
# display(tabla_k.pivot(index='id', columns='combinacion', values='k'))
#
# print('Silueta del K seleccionado por imagen y combinación:')
# display(tabla_k.pivot(index='id', columns='combinacion', values='silueta').round(3))

**Análisis de la selección de K:**

_Escriban aquí su análisis. Preguntas guía: ¿qué K se elige con más frecuencia? ¿cambia el K elegido al aumentar λ/β? ¿qué combinaciones tienen mayor silueta y a qué creen que se debe? (Pista: piensen qué pasa con la silueta en la combinación de solo posición, donde los píxeles forman una rejilla regular.)_

## <a name="7"></a> 7. Construcción de la máscara de la lesión (Actividad 3)

**Qué pide el enunciado:** K-medias agrupa los píxeles pero no dice qué grupos son la lesión. Hay que **diseñar y justificar un criterio automático** que, a partir de características de los grupos (por ejemplo color, tamaño o ubicación), decida cuáles son la lesión y construya una máscara binaria. Pueden comparar dos criterios. Esta actividad vale el 15 % y es la parte más propia del laboratorio.

**Ideas de características que pueden calcular para cada grupo:**
- Color o brillo promedio (por ejemplo, luminancia $0.299R + 0.587G + 0.114B$).
- Tamaño: qué fracción de la imagen ocupa.
- Ubicación: qué tan centrado está, o qué fracción de sus píxeles está cerca del borde.

Ojo: si calculan el color del grupo con el centroide, en la combinación de solo posición (β = 0) el centroide no tiene información de color. Calcularlo con los píxeles originales de la imagen evita ese problema.

**Ideas de criterios (escojan, combinen o propongan otros):**
- Oscuridad: la lesión suele verse más oscura que la piel. _¿Se cumple en sus imágenes?_
- Oscuridad descartando los grupos que están sobre todo en el borde, si en la exploración vieron esquinas oscuras del dermatoscopio.
- Cercanía al centro, si las lesiones suelen estar centradas en la foto.

**Preguntas para justificar:** _¿por qué ese criterio tiene sentido para estas imágenes? ¿cómo van a fijar los umbrales? (solo pueden usar las imágenes de ajuste) ¿qué hace el criterio si ningún grupo lo cumple?_

**Sugerencia:** el código de abajo implementa como ejemplo dos criterios, "oscuridad" y "oscuridad sin borde". Los umbrales (`FRAC_BANDA_BORDE`, `MAX_FRAC_BORDE` y el punto medio de luminancia) son valores iniciales arbitrarios: revisen si tienen sentido con sus imágenes o reemplácenlos por su propio criterio.

_Escriban aquí la descripción y la justificación de sus criterios._

In [ ]:
# FRAC_BANDA_BORDE = 0.1
# MAX_FRAC_BORDE = 0.5
#
#
# def estadisticas_grupos(img, labels):
#     """Calcula tamaño, luminancia, color promedio y fracción en el borde de cada grupo."""
#     alto, ancho = img.shape[:2]
#     color = img.reshape(-1, 3) / 255.0
#     luminancia = color @ np.array([0.299, 0.587, 0.114])
#
#     I, J = np.meshgrid(np.arange(alto), np.arange(ancho), indexing='ij')
#     banda = max(1, round(FRAC_BANDA_BORDE * min(alto, ancho)))
#     en_borde = ((I < banda) | (I >= alto - banda) | (J < banda) | (J >= ancho - banda)).ravel()
#
#     filas = []
#     for g in np.unique(labels):
#         sel = labels == g
#         R, G, B = color[sel].mean(axis=0)
#         filas.append({'grupo': g, 'tamaño': sel.mean(), 'luminancia': luminancia[sel].mean(),
#                       'frac_borde': en_borde[sel].mean(), 'R': R, 'G': G, 'B': B})
#     return pd.DataFrame(filas)
#
#
# def criterio_oscuridad(stats):
#     umbral = (stats['luminancia'].min() + stats['luminancia'].max()) / 2
#     return stats.loc[stats['luminancia'] < umbral, 'grupo'].to_numpy()
#
#
# def criterio_oscuridad_sin_borde(stats):
#     candidatos = stats[stats['frac_borde'] < MAX_FRAC_BORDE]
#     if candidatos.empty:
#         return stats.loc[[(stats['luminancia'] + stats['frac_borde']).idxmin()], 'grupo'].to_numpy()
#     umbral = (candidatos['luminancia'].min() + candidatos['luminancia'].max()) / 2
#     seleccion = candidatos[candidatos['luminancia'] <= umbral]
#     return seleccion['grupo'].to_numpy()
#
#
# CRITERIOS = {'oscuridad': criterio_oscuridad,
#              'oscuridad_sin_borde': criterio_oscuridad_sin_borde}
#
#
# def construir_mascara(img, labels, criterio):
#     """Construye la máscara binaria marcando como lesión los grupos que elige el criterio."""
#     stats = estadisticas_grupos(img, labels)
#     grupos_lesion = CRITERIOS[criterio](stats)
#     return np.isin(labels, grupos_lesion).astype(np.uint8).reshape(img.shape[:2])

**Sugerencia:** antes de aplicar los criterios a todas las imágenes, probarlos en una sola y mirar las características de sus grupos. Para ver los grupos, se puede pintar cada píxel con el color promedio de su grupo, como en la práctica de segmentación.

In [ ]:
# def imagen_segmentada(img, labels):
#     """Pinta cada píxel con el color promedio (original) de su grupo."""
#     color = img.reshape(-1, 3)
#     salida = np.zeros_like(color)
#     for g in np.unique(labels):
#         salida[labels == g] = color[labels == g].mean(axis=0)
#     return salida.reshape(img.shape)
#
#
# def ilustrar_criterios(id_img, c):
#     img = imagenes[id_img]
#     labels = resultados_k[(id_img, c)]['labels']
#     display(estadisticas_grupos(img, labels).round(3))
#
#     paneles = [('Original', img), (f'Grupos {c} (K={resultados_k[(id_img, c)]["k"]})', imagen_segmentada(img, labels))]
#     paneles += [(f'Criterio: {nombre}', construir_mascara(img, labels, nombre)) for nombre in CRITERIOS]
#     paneles += [('Referencia', mascaras_ajuste[id_img])]
#
#     fig, ax = plt.subplots(1, len(paneles), figsize=(3.5 * len(paneles), 3))
#     for a, (titulo, panel) in zip(ax, paneles):
#         a.imshow(panel, cmap='gray')
#         a.set_title(titulo, fontsize=9)
#         a.axis('off')
#     plt.tight_layout()
#     plt.show()
#
#
# ilustrar_criterios(IDS_AJUSTE[0], 'C6')

## <a name="8"></a> 8. Ajuste del método (Actividad 4)

### 8.1 Métricas

**Qué pide el enunciado:** calcular para cada configuración:
- **Dice** (criterio principal): $DICE = \frac{2|X \cap Y|}{|X| + |Y|}$, donde X son los píxeles de lesión en la referencia y Y los estimados. Equivale al F1 a nivel de píxel con la lesión como clase positiva (`f1_score` sobre las máscaras aplanadas).
- **V-measure** (complementaria, `v_measure_score`): compara los **grupos** de K-medias con las etiquetas de referencia.

Para la sección 9 también se piden **precisión** y **recall** con la lesión como clase positiva. _Piensen en qué se diferencian Dice y V-measure: ¿cuál depende del criterio de la máscara y cuál no?_

In [ ]:
# def calcular_metricas(mascara_ref, mascara_est, labels):
#     y_true = mascara_ref.ravel()
#     y_pred = mascara_est.ravel()
#     return {'dice': f1_score(y_true, y_pred, zero_division=0),
#             'precision': precision_score(y_true, y_pred, zero_division=0),
#             'recall': recall_score(y_true, y_pred, zero_division=0),
#             'v_measure': v_measure_score(y_true, labels)}

### 8.2 Verificación de la recomendación de IAG

**Qué pide el enunciado (obligatorio):** **antes** de elegir la configuración final, pedirle a una IAG que recomiende valores de β, λ, K y un criterio para construir la máscara en imágenes dermatoscópicas. En el prompt **no** se pueden anexar imágenes ni mencionar ISIC: hay que describir las imágenes. Luego hay que probar la recomendación con las imágenes de ajuste. El historial va en la sección 12.

_Escriban aquí la recomendación que obtuvieron._

**Sugerencia:** la celda de abajo sirve para probar la recomendación. Llenen `RECOMENDACION_IAG`. Si la IAG da un K fijo, pónganlo en `k`; si dice que se elija con silueta, dejen `k = None`. Si propone un criterio distinto, impleméntenlo como función y agréguenlo a `CRITERIOS`.

In [ ]:
# RECOMENDACION_IAG = {'beta': None, 'lam': None, 'k': None, 'criterio': 'oscuridad'}
#
#
# def segmentar(img, beta, lam, k=None):
#     """Si k es None, selecciona K con la silueta; si no, aplica K-medias con el K indicado."""
#     if k is None:
#         return seleccionar_k(img, beta, lam)
#     pipe.set_params(pesos__beta=beta, pesos__lam=lam, model__n_clusters=k)
#     pipe.fit(img)
#     labels = pipe['model'].labels_
#     X = pipe[:-1].transform(img)
#     return {'k': k, 'labels': labels,
#             'silueta': silhouette_score(X, labels, sample_size=TAMANO_MUESTRA_SILUETA, random_state=SEMILLA)}
#
#
# if RECOMENDACION_IAG['beta'] is None:
#     print('Completar RECOMENDACION_IAG con la respuesta de la IAG antes de ejecutar esta celda.')
# else:
#     filas_iag = []
#     for id_img in IDS_AJUSTE:
#         res = segmentar(imagenes[id_img], RECOMENDACION_IAG['beta'], RECOMENDACION_IAG['lam'], RECOMENDACION_IAG['k'])
#         mascara = construir_mascara(imagenes[id_img], res['labels'], RECOMENDACION_IAG['criterio'])
#         filas_iag.append({'id': id_img, 'k': res['k'], 'silueta': res['silueta'],
#                           **calcular_metricas(mascaras_ajuste[id_img], mascara, res['labels'])})
#     resultados_iag = pd.DataFrame(filas_iag)
#     display(resultados_iag.round(3))
#     print(f"Dice promedio de la recomendación de la IAG: {resultados_iag['dice'].mean():.3f}")

_Escriban aquí cómo le fue a la recomendación de la IAG comparada con sus configuraciones (sección 8.3). La discusión completa va en la sección 12._

### 8.3 Evaluación de todas las configuraciones

**Qué pide el enunciado:** usando **solo** las 15 imágenes de ajuste, evaluar las combinaciones de β y λ y los criterios de máscara. Para cada configuración, calcular Dice y V-measure. El **Dice promedio** es el criterio principal para elegir; el V-measure es complementario.

**Sugerencia:** tratar cada pareja (combinación, criterio) como una configuración y resumirla con el Dice promedio. Además del promedio, la desviación y el mínimo pueden mostrar qué tan estable es una configuración entre imágenes. La tabla y la gráfica de abajo son una forma de presentarlo.

In [ ]:
# filas = []
# for (id_img, c), res in resultados_k.items():
#     for criterio in CRITERIOS:
#         mascara = construir_mascara(imagenes[id_img], res['labels'], criterio)
#         filas.append({'id': id_img, 'combinacion': c, 'criterio': criterio, 'k': res['k'],
#                       'silueta': res['silueta'],
#                       **calcular_metricas(mascaras_ajuste[id_img], mascara, res['labels'])})
# resultados_ajuste = pd.DataFrame(filas)
#
# resumen_ajuste = (resultados_ajuste
#                   .groupby(['combinacion', 'criterio'])
#                   .agg(dice_prom=('dice', 'mean'), dice_std=('dice', 'std'), dice_min=('dice', 'min'),
#                        v_measure_prom=('v_measure', 'mean'), precision_prom=('precision', 'mean'),
#                        recall_prom=('recall', 'mean'), k_prom=('k', 'mean'), silueta_prom=('silueta', 'mean'))
#                   .join(COMBINACIONES[['beta', 'lam', 'descripcion']], on='combinacion')
#                   .sort_values('dice_prom', ascending=False)
#                   .reset_index())
# resumen_ajuste.round(3)

In [ ]:
# fig, ax = plt.subplots(figsize=(12, 4))
# pivote = resumen_ajuste.pivot(index='combinacion', columns='criterio', values='dice_prom').loc[COMBINACIONES.index]
# pivote.plot.bar(ax=ax)
# ax.set_xticklabels([f'{c}\nλ/β={COMBINACIONES.loc[c, "lam/beta"]:g}' for c in pivote.index], rotation=0)
# ax.set(xlabel='Combinación', ylabel='Dice promedio (ajuste)', title='Dice promedio por configuración')
# ax.grid(axis='y')
# plt.tight_layout()
# plt.show()

### 8.4 Visualización de las configuraciones relevantes

**Qué pide el enunciado:** visualizar la imagen original y los grupos obtenidos para las **dos configuraciones con mayor Dice promedio**, y para las de **solo color** y **solo posición**.

**Sugerencia:** para las combinaciones de solo color y solo posición hay que escoger con qué criterio mostrarlas. El código de abajo usa el criterio con el que cada una obtuvo mejor Dice, pero pueden decidir otra cosa. Mostrar también la máscara estimada y la de referencia ayuda a explicar los resultados.

In [ ]:
# def mejor_criterio(combinacion):
#     filas_c = resumen_ajuste[resumen_ajuste['combinacion'] == combinacion]
#     return filas_c.sort_values('dice_prom', ascending=False).iloc[0]['criterio']
#
#
# top2 = resumen_ajuste.head(2)
# configs_visualizar = {f'Top {i + 1}: {f.combinacion}, {f.criterio}': (f.combinacion, f.criterio)
#                       for i, f in enumerate(top2.itertuples())}
# c_color = COMBINACIONES.index[COMBINACIONES['descripcion'] == 'solo color'][0]
# c_posicion = COMBINACIONES.index[COMBINACIONES['descripcion'] == 'solo posición'][0]
# configs_visualizar[f'Solo color: {c_color}'] = (c_color, mejor_criterio(c_color))
# configs_visualizar[f'Solo posición: {c_posicion}'] = (c_posicion, mejor_criterio(c_posicion))
# configs_visualizar

In [ ]:
# def visualizar_configuraciones(ids, configs):
#     """Por cada imagen: original, grupos y máscara estimada de cada configuración, y la máscara de referencia."""
#     cols = 2 + 2 * len(configs)
#     fig, ax = plt.subplots(len(ids), cols, figsize=(1.9 * cols, 1.7 * len(ids)), squeeze=False)
#     for fila, id_img in enumerate(ids):
#         img = imagenes[id_img]
#         ax[fila, 0].imshow(img)
#         ax[fila, 0].set_ylabel(id_img, fontsize=7)
#         ax[fila, -1].imshow(mascaras_ajuste[id_img], cmap='gray')
#         for j, (nombre, (c, criterio)) in enumerate(configs.items()):
#             res = resultados_k[(id_img, c)]
#             mascara = construir_mascara(img, res['labels'], criterio)
#             dice = f1_score(mascaras_ajuste[id_img].ravel(), mascara.ravel(), zero_division=0)
#             ax[fila, 1 + 2 * j].imshow(imagen_segmentada(img, res['labels']))
#             ax[fila, 2 + 2 * j].imshow(mascara, cmap='gray')
#             ax[fila, 2 + 2 * j].set_xlabel(f'K={res["k"]}, Dice={dice:.2f}', fontsize=7)
#             if fila == 0:
#                 ax[fila, 1 + 2 * j].set_title(f'{nombre}\ngrupos', fontsize=7)
#                 ax[fila, 2 + 2 * j].set_title('máscara', fontsize=7)
#         if fila == 0:
#             ax[fila, 0].set_title('Original', fontsize=8)
#             ax[fila, -1].set_title('Referencia', fontsize=8)
#     for a in ax.flat:
#         a.set_xticks([])
#         a.set_yticks([])
#     plt.tight_layout()
#     plt.show()
#
#
# visualizar_configuraciones(IDS_AJUSTE[:5], configs_visualizar)
# visualizar_configuraciones(IDS_AJUSTE[5:10], configs_visualizar)
# visualizar_configuraciones(IDS_AJUSTE[10:], configs_visualizar)

### 8.5 Casos de fallo

**Qué pide el enunciado:** revisar los casos de fallo y usarlos para justificar la elección.

**Sugerencia:** comparar el Dice imagen por imagen entre las configuraciones visualizadas, para ver dónde falla cada una.

In [ ]:
# dice_por_imagen = pd.DataFrame({
#     nombre: resultados_ajuste[(resultados_ajuste['combinacion'] == c) & (resultados_ajuste['criterio'] == criterio)]
#             .set_index('id')['dice']
#     for nombre, (c, criterio) in configs_visualizar.items()
# }).loc[IDS_AJUSTE]
# dice_por_imagen.round(3).style.background_gradient(cmap='RdYlGn', vmin=0, vmax=1)

_Escriban aquí su análisis de los casos de fallo. Preguntas guía: ¿qué tienen en común las imágenes con menor Dice? (lesión más clara que la piel, esquinas oscuras, vellos, lesión que ocupa casi toda la imagen, varias zonas oscuras…) ¿el error viene de falsos positivos (precisión baja) o de falsos negativos (recall bajo)?_

### 8.6 Configuración final

**Qué pide el enunciado:** la configuración final debe incluir β, λ y el criterio de la máscara, justificada con el Dice promedio (criterio principal), el V-measure y los casos de fallo.

**Esta decisión es suya.** La celda de abajo toma por defecto la configuración con mayor Dice promedio. Si después de revisar los casos de fallo prefieren otra (por ejemplo, una con Dice casi igual pero más estable), cámbienla y expliquen por qué.

In [ ]:
# mejor = resumen_ajuste.iloc[0]
# CONFIG_FINAL = {'combinacion': mejor['combinacion'], 'beta': mejor['beta'], 'lam': mejor['lam'],
#                 'criterio': mejor['criterio']}
# dice_ajuste_final = mejor['dice_prom']
#
# print('Configuración final:', CONFIG_FINAL)
# print(f'Dice promedio en ajuste: {dice_ajuste_final:.3f}')
# print(f"V-measure promedio en ajuste: {mejor['v_measure_prom']:.3f}")

**Justificación de la configuración final:**

_Escriban aquí su justificación: valores de β y λ, criterio de máscara, Dice y V-measure promedio, cómo se compara con la recomendación de la IAG y con las otras configuraciones, y qué casos de fallo están aceptando._

## <a name="9"></a> 9. Evaluación sobre las imágenes de prueba (Actividad 5)

**Qué pide el enunciado:**
- Aplicar la configuración final **sin modificaciones** a las 5 imágenes de prueba.
- Una tabla con K elegido, silueta, Dice, V-measure, precisión y recall por imagen (lesión como clase positiva).
- Mostrar la imagen original, la máscara estimada y la de referencia.
- Comparar el Dice promedio de ajuste y de prueba, e identificar los principales aciertos y errores. No se espera un Dice alto en todas.

**Recordatorio de protocolo:** las máscaras de prueba solo se cargan aquí y solo para calcular métricas. Nada de lo que se vea aquí puede usarse para cambiar la configuración.

In [ ]:
# mascaras_prueba = {id_img: cargar_mascara(id_img) for id_img in IDS_PRUEBA}
#
# resultados_prueba, mascaras_est_prueba = [], {}
# for id_img in IDS_PRUEBA:
#     res = segmentar(imagenes[id_img], CONFIG_FINAL['beta'], CONFIG_FINAL['lam'])
#     mascaras_est_prueba[id_img] = construir_mascara(imagenes[id_img], res['labels'], CONFIG_FINAL['criterio'])
#     resultados_prueba.append({'id': id_img, 'k': res['k'], 'silueta': res['silueta'],
#                               **calcular_metricas(mascaras_prueba[id_img], mascaras_est_prueba[id_img], res['labels'])})
#
# tabla_prueba = pd.DataFrame(resultados_prueba).set_index('id')
# tabla_prueba.loc['Promedio'] = tabla_prueba.mean()
# tabla_prueba.round(3)

In [ ]:
# fig, ax = plt.subplots(len(IDS_PRUEBA), 3, figsize=(9, 2.6 * len(IDS_PRUEBA)))
# for fila, id_img in enumerate(IDS_PRUEBA):
#     paneles = [(imagenes[id_img], 'Original', None),
#                (mascaras_est_prueba[id_img], f'Estimada (Dice={tabla_prueba.loc[id_img, "dice"]:.2f})', 'gray'),
#                (mascaras_prueba[id_img], 'Referencia', 'gray')]
#     for a, (panel, titulo, cmap) in zip(ax[fila], paneles):
#         a.imshow(panel, cmap=cmap)
#         a.set_title(f'{id_img} - {titulo}' if cmap is None else titulo, fontsize=8)
#         a.axis('off')
# plt.tight_layout()
# plt.show()

In [ ]:
# comparacion = pd.DataFrame({
#     'Dice promedio': [dice_ajuste_final, tabla_prueba.loc['Promedio', 'dice']],
#     'V-measure promedio': [resumen_ajuste.iloc[0]['v_measure_prom'], tabla_prueba.loc['Promedio', 'v_measure']],
# }, index=['Ajuste (15 imágenes)', 'Prueba (5 imágenes)'])
# comparacion.round(3)

**Análisis de la evaluación:**

_Escriban aquí su análisis. Preguntas guía: ¿cuáles son los principales aciertos y errores en prueba? ¿cómo se compara el Dice con el de ajuste? ¿predomina la precisión baja o el recall bajo en cada imagen?_

## <a name="10"></a> 10. Video (Actividad 6)

Enlace al video (máximo 3 minutos, dirigido a DermAlpes y a un público no especializado): _pegar enlace aquí_

**Lo que el enunciado pide que explique el video:**
1. Qué problema aborda la herramienta.
2. Cómo se obtiene la máscara de la lesión a partir de los grupos de píxeles.
3. Qué muestran los resultados de prueba.
4. En qué casos falla.
5. Qué precauciones deben tenerse al usarla.

_Sugerencia: como el público no es especializado, conviene explicar con ejemplos visuales del notebook y evitar términos como "silueta" o "centroide" sin explicarlos._

## <a name="11"></a> 11. Análisis de resultados

_El enunciado pide responder apoyándose en imágenes, tablas y valores concretos del notebook. Esta sección vale el 30 %._

**Sugerencia:** para las preguntas 1 y 2 hay celdas con ideas de evidencia adicional. Pueden usarlas, cambiarlas o hacer otras.

### 11.1 ¿Qué efecto tuvo incluir la posición de los píxeles? Expliquen en qué imágenes mejoró o perjudicó la segmentación y por qué.

**Sugerencia:** comparar el Dice de cada imagen entre la combinación de solo color y la configuración final, y mirar las imágenes donde la diferencia es más grande.

In [ ]:
# crit_final = CONFIG_FINAL['criterio']
# efecto_posicion = pd.DataFrame({
#     'dice_solo_color': resultados_ajuste[(resultados_ajuste['combinacion'] == c_color) & (resultados_ajuste['criterio'] == crit_final)].set_index('id')['dice'],
#     'dice_config_final': resultados_ajuste[(resultados_ajuste['combinacion'] == CONFIG_FINAL['combinacion']) & (resultados_ajuste['criterio'] == crit_final)].set_index('id')['dice'],
# }).loc[IDS_AJUSTE]
# efecto_posicion['diferencia'] = efecto_posicion['dice_config_final'] - efecto_posicion['dice_solo_color']
# efecto_posicion.sort_values('diferencia').round(3)

_Escriban aquí su respuesta._

### 11.2 ¿Qué relación encontraron entre K, la silueta y el Dice? ¿Por qué estas métricas no necesariamente conducen a las mismas decisiones?

**Sugerencia:** para cada imagen de ajuste, con la configuración final, calcular el Dice que se habría obtenido con cada K y compararlo con la silueta de ese K.

In [ ]:
# filas = []
# for id_img in IDS_AJUSTE:
#     res = resultados_k[(id_img, CONFIG_FINAL['combinacion'])]
#     for k, sil, labels in zip(VALORES_K, res['siluetas'], res['etiquetas']):
#         mascara = construir_mascara(imagenes[id_img], labels, CONFIG_FINAL['criterio'])
#         filas.append({'id': id_img, 'k': k, 'silueta': sil,
#                       'dice': f1_score(mascaras_ajuste[id_img].ravel(), mascara.ravel(), zero_division=0)})
# k_silueta_dice = pd.DataFrame(filas)
#
# fig, ax = plt.subplots(1, 2, figsize=(13, 4))
# k_silueta_dice.groupby('k')[['silueta', 'dice']].mean().plot(marker='o', ax=ax[0])
# ax[0].set(xlabel='K', ylabel='Promedio en ajuste', title='Silueta y Dice promedio por K')
# ax[0].grid()
# ax[1].scatter(k_silueta_dice['silueta'], k_silueta_dice['dice'], c=k_silueta_dice['k'], cmap='viridis')
# ax[1].set(xlabel='Silueta', ylabel='Dice', title='Cada punto es una imagen con un K')
# ax[1].grid()
# plt.colorbar(ax[1].collections[0], ax=ax[1], label='K')
# plt.tight_layout()
# plt.show()
#
# k_por_criterio = k_silueta_dice.loc[k_silueta_dice.groupby('id')['silueta'].idxmax()].set_index('id')[['k', 'dice']]
# k_por_criterio.columns = ['k_silueta', 'dice_con_k_silueta']
# k_mejor_dice = k_silueta_dice.loc[k_silueta_dice.groupby('id')['dice'].idxmax()].set_index('id')[['k', 'dice']]
# k_mejor_dice.columns = ['k_mejor_dice', 'mejor_dice_posible']
# print(f"Correlación silueta-Dice: {k_silueta_dice[['silueta', 'dice']].corr().iloc[0, 1]:.3f}")
# k_por_criterio.join(k_mejor_dice).round(3)

_Escriban aquí su respuesta._ (Ojo: la columna `k_mejor_dice` usa las máscaras de referencia, así que solo sirve para analizar después; el método nunca puede elegir K con ella.)

### 11.3 ¿Cómo se comparan los resultados de ajuste y prueba? ¿Por qué es importante no modificar el método después de consultar las máscaras de prueba?

_Escriban aquí su respuesta, usando la comparación de la sección 9._

### 11.4 Identifiquen dos limitaciones de su modelo y expliquen sus posibles causas a partir de las imágenes y de la representación RGB + posición.

_Escriban aquí su respuesta._

Ideas para revisar contra sus resultados (verifiquen cuáles se cumplen):
- ¿El criterio supone que la lesión es más oscura que la piel?
- RGB mezcla brillo y color: ¿las sombras o la iluminación desigual cambian los grupos?
- ¿La posición favorece grupos compactos o redondeados?
- K-medias tiende a formar grupos convexos de tamaño parecido.
- Vellos, reflejos o marcas.

### 11.5 En el contexto de DermAlpes, ¿qué consecuencias podrían tener los falsos positivos y falsos negativos? ¿Qué sesgos y problemas de generalización deben considerarse antes de utilizar el método con nuevas poblaciones o condiciones de adquisición?

_Escriban aquí su respuesta._

Aspectos que podrían considerar:
- Qué pasa si el tamaño de la lesión se sobreestima o se subestima.
- Zonas relevantes que el especialista deja de mirar.
- Tonos de piel distintos a los de las imágenes con que se construyó el método.
- Otros dermatoscopios, otra iluminación u otra resolución.
- Que el método se construyó con solo 20 imágenes.

## <a name="12"></a> 12. Uso de herramientas de IA generativa

### 12.1 Declaración del uso

_Completar: herramienta(s) utilizada(s) y tipo de uso (ayuda conceptual, generación inicial de código, depuración, redacción, etc.)._

### 12.2 Prompts utilizados

_Completar con los prompts principales._

### 12.3 Actividad de verificación (obligatoria)

**Prompt enviado** (sin imágenes y sin mencionar ISIC; describiendo las imágenes):

> _Pegar aquí el prompt._

**Respuesta y sucesivas interacciones:** _adjuntar el historial completo (texto o enlace a la conversación)._

**Recomendación obtenida:** β = _, λ = _, K = _, criterio = _

**Resultado en las imágenes de ajuste:** Dice promedio = _ (ver sección 8.2).

- ¿La recomendación funcionó en sus imágenes?
- ¿En qué acertó y en qué falló?
- ¿Por qué la IAG no podía anticipar el comportamiento del método en sus imágenes?

### 12.4 Análisis crítico del resultado

_Responder al menos dos: ¿qué partes del contenido generado fueron correctas y útiles? ¿qué errores, imprecisiones o limitaciones se identificaron? ¿qué decisiones técnicas fueron modificadas respecto a la respuesta de la IAG y por qué? ¿qué conceptos del curso permitieron evaluar o mejorar la respuesta generada?_

### 12.5 Aportes propios de los estudiantes

_Completar: qué fue desarrollado, modificado o decidido por los estudiantes, qué ajustes se hicieron sobre el código o explicaciones originales y qué aprendizajes se obtuvieron._

## <a name="13"></a> 13. Nota sobre la reproducibilidad de los resultados

_Completen con los valores que hayan usado, para que alguien más pueda obtener los mismos resultados:_
- Semilla de K-medias y `n_init`: ___
- Tamaño de la muestra y semilla de la silueta: ___
- Tamaño al que se redimensionaron las imágenes e interpolaciones usadas: ___
- Versiones de las librerías: ver la salida de la sección 1.

Para exportar a HTML: `jupyter nbconvert --to html Lab4_DermAlpes.ipynb`.

## <a name="14"></a> 14. Referencias

- Gutman, D., Codella, N. C. F., Celebi, E., Helba, B., Marchetti, M., Mishra, N., & Halpern, A. (2016). *Skin Lesion Analysis toward Melanoma Detection: A Challenge at the International Symposium on Biomedical Imaging (ISBI) 2016, hosted by the International Skin Imaging Collaboration (ISIC)*. arXiv:1605.01397. Conjunto de datos: https://challenge.isic-archive.com/landing/2016/37/
- Scikit-learn: `KMeans`, `silhouette_score`, `f1_score`, `v_measure_score`. https://scikit-learn.org